# Tutorial 5: ML Model
## Introducción
Usaremos *Pima Indians Diabetes* dataset. Vamos a explorar:
1. **Cross-Validation (CV)**
2. **Feature Selection Methods**: SelectKBest, RFE, and SelectFromModel
3. **Hyperparameter Tuning con CV**: GridSearchCV,  RandomizedSearchCV
4. **PyCaret**: AutoML con CV y HT

More information about Kaggle: https://www.kaggle.com/

In [ ]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

# Cargar datasets
dset_trn = pd.read_csv('pima_train.csv')
dset_tst = pd.read_csv('pima_test.csv')

print('Forma del set de entrenamiento:', dset_trn.shape)
print('Forma del set de test:', dset_tst.shape)

## Pre procesamiento  de datos

Datos faltantes??

In [ ]:
# Verificar valores faltantes
print('Missing values (proportions):')
print(dset_trn.isna().sum() / dset_trn.shape[0])

# Eliminar Insulin (demasiados valores faltantes)
dset_trn.drop(columns='Insulin', inplace=True)
dset_tst.drop(columns='Insulin', inplace=True)

# Preparar datos
X_train = dset_trn.drop(columns=['Id','Outcome']).to_numpy()
X_test = dset_tst.drop(columns='Id').to_numpy()
y_train = dset_trn['Outcome'].to_numpy()

# Imputar valores faltantes
from sklearn.impute import SimpleImputer
imp = SimpleImputer(strategy='mean').fit(X_train)
X_train = imp.transform(X_train)
X_test = imp.transform(X_test)

# Escalar datos
from sklearn import preprocessing
mm_scaler = preprocessing.StandardScaler()
X_train = mm_scaler.fit_transform(X_train)
X_test = mm_scaler.transform(X_test)

print('\nForma de datos procesados:')
print(f'X_train: {X_train.shape}, y_train: {y_train.shape}')
print(f'X_test: {X_test.shape}')

---
# PARTE 1: ENTENDIENDO CROSS-VALIDATION (CV)

## ¿Qué es Cross-Validation?

Cross-Validation (CV) es una técnica para evaluar la capacidad de generalización de un modelo.
En lugar de usar un único train/test split, dividimos el set de entrenamiento en múltiples 'folds' (pliegues).

### ¿Por qué es importante para hyperparameter tuning?

**Sin CV (incorrecto):**
```
Train Set → Tune hyperparameters → Evaluate en Train Set
                                    ↑ DATA LEAKAGE!
```

**Con CV (correcto):**
```
Train Set → Split en 5 folds
  - Fold 1: Train en folds 2-5, evalúa en fold 1
  - Fold 2: Train en folds 1,3-5, evalúa en fold 2
  - ... etc
  → Mejor estimación de rendimiento real
```

## Ejemplo 1: CV sin Hyperparameter Tuning

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold

# Crear modelo con hyperparametros fijos
lr = LogisticRegression(max_iter=1000, C=1.0)

# Usar CV para evaluar el modelo
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(lr, X_train, y_train, cv=skf, scoring='roc_auc')

print('Puntuaciones AUC para cada fold:')
print(cv_scores)
print(f'\nAUC promedio: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})')

## Ejemplo 2: CV CON Hyperparameter Tuning 

In [ ]:
from sklearn.model_selection import GridSearchCV

# Definir grilla de hiperparámetros
param_grid = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'penalty': ['l2'],
    'solver': ['lbfgs']
}

# GridSearchCV realiza CV automáticamente
grid_search = GridSearchCV(
    estimator=LogisticRegression(max_iter=1000),
    param_grid=param_grid,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print('Mejores hiperparámetros:')
print(grid_search.best_params_)
print(f'Mejor puntuación CV AUC: {grid_search.best_score_:.4f}')
print(f'\nResultados de validación cruzada:')
cv_results = pd.DataFrame(grid_search.cv_results_)
print(cv_results[['param_C', 'mean_test_score', 'std_test_score']].head(10))

## Ejemplo 3: Visualizando CV 

In [ ]:
# Comparación: Puntuación de entrenamiento vs Puntuación CV
best_model = grid_search.best_estimator_

# Puntuación en el set de entrenamiento COMPLETO
train_score = best_model.score(X_train, y_train)

# CV score 
cv_score = grid_search.best_score_

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Gráfico 1: Entrenamiento vs Puntuación CV
scores = [train_score, cv_score]
labels = ['Training Score', 'CV Score']
colors = ['#ff6b6b', '#51cf66']
axes[0].bar(labels, scores, color=colors, alpha=0.7)
axes[0].set_ylabel('AUC Score')
axes[0].set_title('Training vs CV Score')
axes[0].set_ylim([0, 1])
for i, v in enumerate(scores):
    axes[0].text(i, v + 0.02, f'{v:.4f}', ha='center', fontweight='bold')

# Gráfico 2: Puntuaciones CV por hiperparámetro C
cv_results_df = pd.DataFrame(grid_search.cv_results_)
axes[1].errorbar(cv_results_df['param_C'], 
                 cv_results_df['mean_test_score'],
                 yerr=cv_results_df['std_test_score'],
                 marker='o', capsize=5, capthick=2)
axes[1].set_xlabel('C (Regularization parameter)')
axes[1].set_ylabel('Mean CV AUC')
axes[1].set_title('Hyperparameter Tuning with CV')
axes[1].set_xscale('log')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()



---
# PARTE 2: SELECCIÓN DE ATRIBUTOS (Feature Selection)

## ¿Por qué seleccionar atributos?
- Reducir overfitting
- Mejorar eficiencia computacional
- Interpretabilidad
- Pueden mejorar el rendimiento del modelo

## Métodos a explorar:
1. **SelectKBest**: Selecciona los K atributos con mejor score estadístico
2. **RFE (Recursive Feature Elimination)**: Elimina atributos recursivamente
3. **SelectFromModel**: Selecciona atributos basado en importancia del modelo

## 1. SelectKBest

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif

# SelectKBest con criterio f_classif (Valor F de ANOVA)
selector_kbest = SelectKBest(score_func=f_classif, k=4)
X_train_kbest = selector_kbest.fit_transform(X_train, y_train)
X_test_kbest = selector_kbest.transform(X_test)

# Ver cuáles atributos fueron seleccionados
feature_names = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 
                 'BMI', 'DiabetesPedigreeFunction', 'Age']
selected_mask = selector_kbest.get_support()
selected_features = [f for f, selected in zip(feature_names, selected_mask) if selected]

print('Resultados de SelectKBest:')
print(f'Features seleccionados: {selected_features}')
print(f'Forma: {X_train_kbest.shape[0]} muestras, {X_train_kbest.shape[1]} features')

# Mostrar puntuaciones de cada atributo
scores_df = pd.DataFrame({
    'Feature': feature_names,
    'F-Score': selector_kbest.scores_,
    'Selected': selected_mask
}).sort_values('F-Score', ascending=False)

print('\nPuntuaciones de características:')
print(scores_df)

## 2. Recursive Feature Elimination (RFE)

In [ ]:
from sklearn.feature_selection import RFE

# RFE: Elimina recursivamente atributos usando un estimador
estimator = LogisticRegression(max_iter=1000, random_state=42)
rfe = RFE(estimator=estimator, n_features_to_select=4, step=1)

X_train_rfe = rfe.fit_transform(X_train, y_train)
X_test_rfe = rfe.transform(X_test)

# Ver cuáles atributos fueron seleccionados
selected_rfe = [f for f, selected in zip(feature_names, rfe.support_) if selected]

print('Resultados de RFE:')
print(f'Selected features: {selected_rfe}')
print(f'Shape: {X_train_rfe.shape[0]} samples, {X_train_rfe.shape[1]} features')

# Mostrar ranking
rfe_df = pd.DataFrame({
    'Feature': feature_names,
    'Ranking': rfe.ranking_,
    'Selected': rfe.support_
}).sort_values('Ranking')

print('\nRanking de características (1 = seleccionado):')
print(rfe_df)

## 3. SelectFromModel (basado en importancia del modelo)

In [ ]:
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestClassifier

# SelectFromModel usa importancia de features de un modelo
rf = RandomForestClassifier(n_estimators=100, random_state=42)
selector_model = SelectFromModel(rf, threshold='median')

X_train_sfm = selector_model.fit_transform(X_train, y_train)
X_test_sfm = selector_model.transform(X_test)

# Ver cuáles atributos fueron seleccionados
selected_sfm = [f for f, selected in zip(feature_names, selector_model.get_support()) if selected]

print('Resultados de SelectFromModel:')
print(f'Selected features: {selected_sfm}')
print(f'Shape: {X_train_sfm.shape[0]} samples, {X_train_sfm.shape[1]} features')

# Mostrar importancia de características
rf_fitted = selector_model.estimator_
sfm_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': rf_fitted.feature_importances_,
    'Selected': selector_model.get_support()
}).sort_values('Importance', ascending=False)

print('\nImportancia de características (Random Forest):')
print(sfm_df)

## Comparación de los 3 métodos

In [ ]:
# Comparar qué atributos selecciona cada método
comparison_df = pd.DataFrame({
    'Feature': feature_names,
    'SelectKBest': selector_kbest.get_support(),
    'RFE': rfe.support_,
    'SelectFromModel': selector_model.get_support()
})

print('\nComparación de Métodos de Selección de Características:')
print(comparison_df.to_string(index=False))

# Visualizar
fig, ax = plt.subplots(figsize=(10, 5))

x = np.arange(len(feature_names))
width = 0.25

ax.bar(x - width, selector_kbest.get_support().astype(int), width, label='SelectKBest')
ax.bar(x, rfe.support_.astype(int), width, label='RFE')
ax.bar(x + width, selector_model.get_support().astype(int), width, label='SelectFromModel')

ax.set_xlabel('Features')
ax.set_ylabel('Selected (1) / Not Selected (0)')
ax.set_title('Feature Selection Methods Comparison')
ax.set_xticks(x)
ax.set_xticklabels(feature_names, rotation=45, ha='right')
ax.legend()
ax.set_ylim([0, 1.2])

plt.tight_layout()
plt.show()

## Evaluar modelos con cada método de Feature Selection usando CV

## Pipeline en scikit-learn
Pipeline encadena múltiples pasos de procesamiento en un solo objeto. Cada paso transforma los datos y pasa el resultado al siguiente. El último paso es un estimador (modelo).
```python
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ('nombre_paso1', transformador1),
    ('nombre_paso2', transformador2),
    ('nombre_modelo', estimador)
])
```

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
# Evaluar modelos con CV
results_fs = {}

# Sin feature selection
lr_none = LogisticRegression(max_iter=1000, random_state=42)
scores_none = cross_val_score(lr_none, X_train, y_train, cv=5, scoring='roc_auc')
results_fs['No Feature Selection'] = scores_none.mean()

# Con SelectKBest
pipeline_kbest = Pipeline([
    ('selector', SelectKBest(f_classif, k=4)),
    ('model', LogisticRegression(max_iter=1000, random_state=42))
])
scores_kbest = cross_val_score(pipeline_kbest, X_train, y_train, cv=5, scoring='roc_auc')
results_fs['SelectKBest'] = scores_kbest.mean()

#data leakage por que? 
#lr_kbest = LogisticRegression(max_iter=1000, random_state=42)
#scores_kbest = cross_val_score(lr_kbest, X_train_kbest, y_train, cv=5, scoring='roc_auc')
#results_fs['SelectKBest'] = scores_kbest.mean()

# Con RFE
pipeline_rfe = Pipeline([
    ('selector', RFE(estimator=LogisticRegression(max_iter=1000), n_features_to_select=4, step=1)),
    ('model', LogisticRegression(max_iter=1000, random_state=42))])
scores_rfe = cross_val_score(pipeline_rfe, X_train, y_train, cv=5, scoring='roc_auc')
results_fs['RFE'] = scores_rfe.mean()

# Con SelectFromModel
pipeline_sfm = Pipeline([
    ('selector', SelectFromModel(RandomForestClassifier(n_estimators=100, random_state=42), threshold='median')),
    ('model', LogisticRegression(max_iter=1000, random_state=42))
])
scores_sfm = cross_val_score(pipeline_sfm, X_train, y_train, cv=5, scoring='roc_auc')
results_fs['SelectFromModel'] = scores_sfm.mean()

print('Puntuaciones CV AUC para cada Método de Selección de Características:')
for method, score in sorted(results_fs.items(), key=lambda x: x[1], reverse=True):
    print(f'{method:30s}: {score:.4f}')

# Visualizar
fig, ax = plt.subplots(figsize=(10, 5))
methods = list(results_fs.keys())
scores = list(results_fs.values())
colors_fs = ['#74b9ff' if s == max(scores) else '#a29bfe' for s in scores]

bars = ax.barh(methods, scores, color=colors_fs, alpha=0.8)
ax.set_xlabel('Mean CV AUC Score')
ax.set_title('Feature Selection Methods Performance Comparison')
ax.set_xlim([0.7, 0.9])

for i, (bar, score) in enumerate(zip(bars, scores)):
    ax.text(score + 0.005, i, f'{score:.4f}', va='center', fontweight='bold')

plt.tight_layout()
plt.show()

---
# PARTE 3: MODELOS CLÁSICOS CON CV Y FEATURE SELECTION

## Logistic Regression con CV

In [ ]:
from sklearn.linear_model import LogisticRegressionCV

# LogisticRegressionCV automáticamente ajusta C con CV
lr_cv = LogisticRegressionCV(
    penalty='l2',
    scoring='roc_auc',
    solver='lbfgs',
    Cs=np.logspace(-4, 4, 20),
    cv=5,
    max_iter=1000,
    random_state=42,
    n_jobs=-1
)

lr_cv.fit(X_train, y_train)

print('Resultados de LogisticRegressionCV:')
print(f'Mejor C: {lr_cv.C_[0]:.6f}')
print(f'AUC CV promedio: {lr_cv.scores_[1].mean():.4f}')

y_pred_lr = lr_cv.predict_proba(X_test)[:, 1]
print(f'Predicciones de test - Mín: {y_pred_lr.min():.4f}, Máx: {y_pred_lr.max():.4f}')

## Support Vector Machine (SVM) con CV 

In [ ]:
from sklearn.svm import SVC

svm_params = {
    'C': [0.1, 1, 10, 100],
    'kernel': ['linear', 'rbf'],
    'gamma': [0.001, 0.01, 0.1, 1]
}

svm_grid = GridSearchCV(
    SVC(probability=True),
    param_grid=svm_params,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1
)

svm_grid.fit(X_train, y_train)

print('Resultados de SVM:')
print(f'Mejores parámetros: {svm_grid.best_params_}')
print(f'Mejor CV AUC: {svm_grid.best_score_:.4f}')

y_pred_svm = svm_grid.predict_proba(X_test)[:, 1]
print(f'Test predictions - Min: {y_pred_svm.min():.4f}, Max: {y_pred_svm.max():.4f}')

## Random Forest con CV

In [ ]:
rf_params = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, 15],
    'min_samples_leaf': [2, 4, 8]
}

rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid=rf_params,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1
)

rf_grid.fit(X_train, y_train)

print('Resultados de Random Forest:')
print(f'Best parameters: {rf_grid.best_params_}')
print(f'Best CV AUC: {rf_grid.best_score_:.4f}')

y_pred_rf = rf_grid.predict_proba(X_test)[:, 1]
print(f'Test predictions - Min: {y_pred_rf.min():.4f}, Max: {y_pred_rf.max():.4f}')

## Gradient Boosting con CV

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

gbm_params = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 5, 7]
}

gbm_grid = GridSearchCV(
    GradientBoostingClassifier(random_state=42),
    param_grid=gbm_params,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1
)

gbm_grid.fit(X_train, y_train)

print('Resultados de Gradient Boosting:')
print(f'Best parameters: {gbm_grid.best_params_}')
print(f'Best CV AUC: {gbm_grid.best_score_:.4f}')

y_pred_gbm = gbm_grid.predict_proba(X_test)[:, 1]
print(f'Test predictions - Min: {y_pred_gbm.min():.4f}, Max: {y_pred_gbm.max():.4f}')

---
# PARTE 4: PYCARET - AUTOMATED MACHINE LEARNING

## ¿Qué es PyCaret?
PyCaret es una librería de ML automatizada que:
- Realiza preprocesamiento automático
- Tunea automáticamente hyperparameters usando CV
- Prueba múltiples algoritmos
- Genera reportes y visualizaciones


In [ ]:

from pycaret.classification import setup, compare_models,pull,plot_model
from pycaret.classification import *
pycaret_available = True

## PyCaret: Setup y Comparación automática de modelos

In [ ]:
# Preparar datos
df_train_pycaret = dset_trn.copy()
df_train_pycaret = df_train_pycaret.drop(columns=['Id'])

print('\nSETUP: Preprocesamiento automático')

# Setup realiza preprocesamiento
exp1 = setup(
    data=df_train_pycaret,
    target='Outcome',
    session_id=42,
    verbose=False,
    #feature_selection = True, ## usar este argumento para usar la seleccion clasica (foward) de argumentos
    remove_multicollinearity = True, multicollinearity_threshold = 0.3    #usar este argumento para no permitir multicollinearity
)
print('Configuración completada')

In [ ]:
### Imprimir data inicial
exp1.train

In [ ]:
### Imprimir data posterior a la selección de atributos
exp1.train_transformed

## PyCaret: Comparar múltiples modelos automáticamente

In [ ]:
print('\nCOMPARAR MODELOS: Probando múltiples algoritmos con CV')
best_model_pycaret = compare_models(
        fold=5,
        sort='AUC')
print(f'\nMejor modelo: {type(best_model_pycaret).__name__}')

In [ ]:
plot_model(best_model_pycaret, plot = 'confusion_matrix')
plot_model(best_model_pycaret, plot = 'class_report')

## PyCaret: Tuning del mejor modelo

In [ ]:
print('\nAJUSTAR MODELO: Ajuste fino con Optimización Bayesiana')
tuned_model = tune_model(
        best_model_pycaret,
        optimize='AUC',
        fold=5,
        n_iter=10)
print('Ajuste completado')

### Usar decision trees

In [ ]:
dt = create_model('dt')

In [ ]:
dt_grid = {'max_depth' : [None, 2, 4, 6, 8, 10, 12]}

# tune model with custom grid and metric = F1
tuned_dt = tune_model(dt, custom_grid = dt_grid, optimize = 'F1')

##### también se usa automaticamente como:
#tuned_dt = tune_model(dt)

In [ ]:
# ensamble con bagging
ensemble_model(dt, method = 'Bagging')

In [ ]:
# ensamble con boosting
ensemble_model(dt, method = 'Boosting')

In [ ]:
# ensamble con stack
best_recall_models_top3 = compare_models(sort = 'Recall', n_select = 3)
stack=stack_models(best_recall_models_top3)
stack

In [ ]:
## Iterpretabilidad del modelo boosting con Shap values!!!
lightgbm = create_model('lightgbm')
interpret_model(lightgbm, plot = 'summary')

## PyCaret: Finalizando y predicciones: Ejercicio
Use df_test_pycaret y desarrolle la predicción de este nuevo conjunto de datos. Compare sus resultados con prima_solution.csv. Desarrolle el mismo ejercicio con diferentes modelos y ensambles hasta obtener el mejor modelo!!


In [ ]:
df_test_pycaret = dset_tst.copy()
df_test_pycaret = df_test_pycaret.drop(columns='Id')

In [ ]:
print('\nFINALIZAR MODELO: Entrenamiento con todos los datos')
final_model = finalize_model(tuned_model)


print('Realizando predicciones en el conjunto de prueba...')
predictions = predict_model(final_model, data = df_test_pycaret)
predictions.head()
